# PCOS Dataset Modeling (SMOTE focus)
This notebook extracts and reuses the PCOS preprocessing and modeling logic from `HOPEFUL3 (1).ipynb`, limiting execution to the PCOS dataset and adding SMOTE-based training metrics (accuracy %, F1).


In [68]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from imblearn.over_sampling import SMOTE

# Load PCOS dataset (path mirrors the original notebook)
df_pcos = pd.read_csv('pcos_dataset.csv')
print('### Raw PCOS data - first 5 rows')
print(df_pcos.head())
print('\n### Raw PCOS data info')
df_pcos.info()


### Raw PCOS data - first 5 rows
    Age   BMI  Menstrual_Irregularity  Testosterone_Level(ng/dL)  \
0  24.0  34.7                     1.0                       25.2   
1  37.0  26.4                     0.0                       57.1   
2  32.0  23.6                     0.0                       92.7   
3  28.0  28.8                     0.0                       63.1   
4  25.0  22.1                     1.0                       59.8   

   Antral_Follicle_Count  PCOS_Diagnosis  Unnamed: 6  Unnamed: 7  Unnamed: 8  \
0                   20.0             0.0         NaN         NaN         NaN   
1                   25.0             0.0         NaN         NaN         NaN   
2                   28.0             0.0         NaN         NaN         NaN   
3                   26.0             0.0         NaN         NaN         NaN   
4                    8.0             0.0         NaN         NaN         NaN   

   Unnamed: 9  ...  Unnamed: 15  Unnamed: 16  Unnamed: 17  Unnamed: 18  \
0  

In [69]:
# Preprocess: drop identifier and empty columns, drop NaN targets, then impute numerics
print('### Preprocessing PCOS data (drop id, empty cols, NaN targets; impute numerics)')
# Drop id if present
df_pcos_processed = df_pcos.drop('id', axis=1, errors='ignore')

# Drop columns that are entirely NaN (e.g., unnamed columns)
all_nan_cols = [c for c in df_pcos_processed.columns if df_pcos_processed[c].isna().all()]
if all_nan_cols:
    df_pcos_processed = df_pcos_processed.drop(columns=all_nan_cols)
    print(f'Dropped all-NaN columns: {all_nan_cols}')

# Drop rows where target is NaN, then cast target to int
if 'PCOS_Diagnosis' in df_pcos_processed.columns:
    before_rows = len(df_pcos_processed)
    df_pcos_processed = df_pcos_processed.dropna(subset=['PCOS_Diagnosis'])
    after_rows = len(df_pcos_processed)
    if before_rows != after_rows:
        print(f'Dropped {before_rows - after_rows} rows with NaN target')
    df_pcos_processed['PCOS_Diagnosis'] = df_pcos_processed['PCOS_Diagnosis'].astype(int)

# Impute numeric columns with their mean
num_cols = df_pcos_processed.select_dtypes(include=[np.number]).columns
for col in num_cols:
    if df_pcos_processed[col].isna().any():
        df_pcos_processed[col] = df_pcos_processed[col].fillna(df_pcos_processed[col].mean())

print(df_pcos_processed.head())
print('\n### Processed info')
df_pcos_processed.info()


### Preprocessing PCOS data (drop id, empty cols, NaN targets; impute numerics)
Dropped all-NaN columns: ['Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24']
Dropped 7 rows with NaN target
    Age   BMI  Menstrual_Irregularity  Testosterone_Level(ng/dL)  \
0  24.0  34.7                     1.0                       25.2   
1  37.0  26.4                     0.0                       57.1   
2  32.0  23.6                     0.0                       92.7   
3  28.0  28.8                     0.0                       63.1   
4  25.0  22.1                     1.0                       59.8   

   Antral_Follicle_Count  PCOS_Diagnosis  
0                   20.0               0  
1                   25.0               0  
2                   28.0               0  
3

In [70]:
# Train/test split with scaling (same approach as source notebook)
X = df_pcos_processed.drop('PCOS_Diagnosis', axis=1)
y = df_pcos_processed['PCOS_Diagnosis']

# Safety: ensure no NaNs remain
X = X.fillna(X.mean())

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)
print('y_train shape:', y_train.shape)
print('y_test shape:', y_test.shape)
print('\nScaled features (head):')
print(X_scaled.head())


X_train shape: (800, 5)
X_test shape: (200, 5)
y_train shape: (800,)
y_test shape: (200,)

Scaled features (head):
        Age       BMI  Menstrual_Irregularity  Testosterone_Level(ng/dL)  \
0 -0.918642  1.685157                0.941697                  -1.510220   
1  0.618141  0.002635               -1.061913                  -0.132168   
2  0.027071 -0.564962               -1.061913                   1.405721   
3 -0.445785  0.489148               -1.061913                   0.127027   
4 -0.800428 -0.869033                0.941697                  -0.015530   

   Antral_Follicle_Count  
0               0.358206  
1               1.065844  
2               1.490426  
3               1.207371  
4              -1.340124  


In [71]:
# Baseline models (without SMOTE) mirroring the original notebook
models = {
    'SVM': SVC(random_state=42),
    'NeuralNet': MLPClassifier(random_state=42, max_iter=500, early_stopping=True),
    'RandomForest': RandomForestClassifier(random_state=42)
}

for name, model in models.items():
    print(f'\n### {name} (no SMOTE)')
    model.fit(X_train, y_train)
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    train_acc = accuracy_score(y_train, y_pred_train)
    test_acc = accuracy_score(y_test, y_pred_test)
    train_f1 = f1_score(y_train, y_pred_train)
    test_f1 = f1_score(y_test, y_pred_test)
    print(f'Train accuracy: {train_acc:.4f}, F1: {train_f1:.4f}')
    print(f'Test accuracy:  {test_acc:.4f}, F1: {test_f1:.4f}')
    print('Confusion matrix (test):')
    print(confusion_matrix(y_test, y_pred_test))




### SVM (no SMOTE)
Train accuracy: 0.9788, F1: 0.9470
Test accuracy:  0.9550, F1: 0.8831
Confusion matrix (test):
[[157   4]
 [  5  34]]

### NeuralNet (no SMOTE)
Train accuracy: 0.9375, F1: 0.8227
Test accuracy:  0.9150, F1: 0.7606
Confusion matrix (test):
[[156   5]
 [ 12  27]]

### RandomForest (no SMOTE)
Train accuracy: 1.0000, F1: 1.0000
Test accuracy:  0.9900, F1: 0.9737
Confusion matrix (test):
[[161   0]
 [  2  37]]


In [72]:
# SMOTE + RandomForest evaluation (adds requested SMOTE prediction results)
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

rf_smote = RandomForestClassifier(random_state=42)
rf_smote.fit(X_train_smote, y_train_smote)

y_pred_smote = rf_smote.predict(X_test)
acc_smote = accuracy_score(y_test, y_pred_smote)
f1_smote = f1_score(y_test, y_pred_smote)

print('### SMOTE + RandomForest results (test set)')
print(f'Accuracy: {acc_smote*100:.2f}%')
print(f'F1 score: {f1_smote:.4f}')
print('Confusion matrix (test):')
print(confusion_matrix(y_test, y_pred_smote))
print('\nSample predictions (first 20):')
print(y_pred_smote[:20])


### SMOTE + RandomForest results (test set)
Accuracy: 99.50%
F1 score: 0.9870
Confusion matrix (test):
[[161   0]
 [  1  38]]

Sample predictions (first 20):
[0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0]


In [73]:
# SMOTE applied to all three models (mirroring original summary)
smote_all = SMOTE(random_state=42)
X_train_smote_all, y_train_smote_all = smote_all.fit_resample(X_train, y_train)

models_smote = {
    'SVM_SMOTE': SVC(random_state=42),
    'NeuralNet_SMOTE': MLPClassifier(random_state=42, max_iter=500, early_stopping=True),
    'RandomForest_SMOTE': RandomForestClassifier(random_state=42)
}

for name, model in models_smote.items():
    print(f"\n### {name}")
    model.fit(X_train_smote_all, y_train_smote_all)
    y_pred_test = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred_test)
    f1 = f1_score(y_test, y_pred_test)
    print(f"Test accuracy: {acc:.4f}, F1: {f1:.4f}")
    print("Confusion matrix (test):")
    print(confusion_matrix(y_test, y_pred_test))



### SVM_SMOTE
Test accuracy: 0.9600, F1: 0.9048
Confusion matrix (test):
[[154   7]
 [  1  38]]

### NeuralNet_SMOTE
Test accuracy: 0.9050, F1: 0.7865
Confusion matrix (test):
[[146  15]
 [  4  35]]

### RandomForest_SMOTE
Test accuracy: 0.9950, F1: 0.9870
Confusion matrix (test):
[[161   0]
 [  1  38]]


In [74]:
# Hyperparameter tuning with SMOTE for SVM, NeuralNet, RandomForest
from sklearn.model_selection import GridSearchCV

# Use the SMOTE-resampled training set
tuned_results = {}

# SVM tuning
tuned_svm_params = {'C': [10, 100], 'kernel': ['rbf']}
tuned_svm = GridSearchCV(SVC(random_state=42), tuned_svm_params, cv=3, scoring='f1', n_jobs=-1)
tuned_svm.fit(X_train_smote_all, y_train_smote_all)
y_pred_svm_tuned = tuned_svm.predict(X_test)
tuned_results['SVM_TUNED_SMOTE'] = {
    'best_params': tuned_svm.best_params_,
    'acc': accuracy_score(y_test, y_pred_svm_tuned),
    'precision': precision_score(y_test, y_pred_svm_tuned),
    'recall': recall_score(y_test, y_pred_svm_tuned),
    'f1': f1_score(y_test, y_pred_svm_tuned),
    'cm': confusion_matrix(y_test, y_pred_svm_tuned)
}

# NeuralNet tuning
tuned_nn_params = {
    'hidden_layer_sizes': [(100,), (100, 50)],
    'activation': ['relu'],
    'solver': ['adam'],
    'alpha': [0.0001]
}
tuned_nn = GridSearchCV(MLPClassifier(random_state=42, max_iter=500, early_stopping=True), tuned_nn_params, cv=3, scoring='f1', n_jobs=-1)
tuned_nn.fit(X_train_smote_all, y_train_smote_all)
y_pred_nn_tuned = tuned_nn.predict(X_test)
tuned_results['NN_TUNED_SMOTE'] = {
    'best_params': tuned_nn.best_params_,
    'acc': accuracy_score(y_test, y_pred_nn_tuned),
    'precision': precision_score(y_test, y_pred_nn_tuned),
    'recall': recall_score(y_test, y_pred_nn_tuned),
    'f1': f1_score(y_test, y_pred_nn_tuned),
    'cm': confusion_matrix(y_test, y_pred_nn_tuned)
}

# RandomForest tuning
tuned_rf_params = {
    'n_estimators': [100],
    'max_features': ['sqrt'],
    'min_samples_leaf': [1]
}
tuned_rf = GridSearchCV(RandomForestClassifier(random_state=42), tuned_rf_params, cv=3, scoring='f1', n_jobs=-1)
tuned_rf.fit(X_train_smote_all, y_train_smote_all)
y_pred_rf_tuned = tuned_rf.predict(X_test)
tuned_results['RF_TUNED_SMOTE'] = {
    'best_params': tuned_rf.best_params_,
    'acc': accuracy_score(y_test, y_pred_rf_tuned),
    'precision': precision_score(y_test, y_pred_rf_tuned),
    'recall': recall_score(y_test, y_pred_rf_tuned),
    'f1': f1_score(y_test, y_pred_rf_tuned),
    'cm': confusion_matrix(y_test, y_pred_rf_tuned),
    'feature_importances': pd.Series(tuned_rf.best_estimator_.feature_importances_, index=X.columns)
}

# Print tuned results
for name, res in tuned_results.items():
    print(f"\n### {name}")
    print(f"Best params: {res['best_params']}")
    print(f"Test Accuracy: {res['acc']:.4f}")
    print(f"Precision: {res['precision']:.4f}, Recall: {res['recall']:.4f}, F1: {res['f1']:.4f}")
    print("Confusion matrix (test):")
    print(res['cm'])
    if 'feature_importances' in res:
        print("\nFeature importances (RF tuned):")
        print(res['feature_importances'].sort_values(ascending=False))




### SVM_TUNED_SMOTE
Best params: {'C': 100, 'kernel': 'rbf'}
Test Accuracy: 0.9750
Precision: 1.0000, Recall: 0.8718, F1: 0.9315
Confusion matrix (test):
[[161   0]
 [  5  34]]

### NN_TUNED_SMOTE
Best params: {'activation': 'relu', 'alpha': 0.0001, 'hidden_layer_sizes': (100, 50), 'solver': 'adam'}
Test Accuracy: 0.9400
Precision: 0.8293, Recall: 0.8718, F1: 0.8500
Confusion matrix (test):
[[154   7]
 [  5  34]]

### RF_TUNED_SMOTE
Best params: {'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 100}
Test Accuracy: 0.9950
Precision: 1.0000, Recall: 0.9744, F1: 0.9870
Confusion matrix (test):
[[161   0]
 [  1  38]]

Feature importances (RF tuned):
Menstrual_Irregularity       0.374420
BMI                          0.330038
Testosterone_Level(ng/dL)    0.163221
Antral_Follicle_Count        0.125228
Age                          0.007093
dtype: float64
